In [1]:
import os
os.getcwd()

'/Users/vigneshsomjit/Git/Bayes-Inv-Prob/bip/notebooks'

In [2]:
# Add path to param.py. Current working directory assumed to be .../bip/notebooks

import sys
sys.path.append("..") # Go to parent directory 
from param import *

# Running Example
Let's suppose we are interested in the parameters $\theta \in [0,\infty)$ and $\Sigma \in \mathbb{R}^{2 \times 2}$. The former is a scalar constrained to be nonnegative and the latter is a positive semidefinite (PSD) matrix. We will create a `ParamGroup` object that encodes the parameter set $\{\theta, \Sigma\}$.

# ParamInfo Class
We start by defining two `ParamInfo` objects, one for $\theta$ and one for $\Sigma$. These objects will store the value type, shape, and constraints for each parameter. The value type is the Python type of the individual elements of the parameter. These attributes should be treated as immutable; no setter is implemented for them. They are set when instantiating the object. This is done to avoid problems with `ParamGroup` and `ParamValue` (see below), where a user decides to update parameter info that might be in violation of existing parameter values. 

In [3]:
theta_info = ParamInfo(value_type="float", shape=(), constraint=(0, None))
Sigma_info = ParamInfo(value_type="float", shape=(2,2), constraint="psd")

In [4]:
# Get length of parameter, the number of scalar values that make up the parameter.

theta_len = len(theta_info)
Sigma_len = len(Sigma_info)

print(f"theta length: {theta_len}")
print(f"Sigma length: {Sigma_len}")

theta length: 1
Sigma length: 4


In [5]:
# Print summary of parameters.

print("theta:")
print(theta_info)

print("\nSigma:")
print(Sigma_info)

theta:
ParamInfo<value_type: float, shape: (), constraint: (0, None), length: 1>

Sigma:
ParamInfo<value_type: float, shape: (2, 2), constraint: psd, length: 4>


## Validating values against the parameter information
We have not yet discussed parameter values (see `ParamGroupValues` below), but for now we note that the `ParamInfo` class
contains a method `validate_value` that a given array satisfies the requirements imposed by the `value_type`, `shape`, and `constraint` attributes in a `ParamInfo` object. At present, the array must always be a `numpy.ndarray`.

In [6]:
# Valid values for theta.
theta_values = [1.0, 2.5, 6.0]

for val in theta_values:
    theta_info.validate_value(np.array(val))

In [7]:
# Currently, infinite values are treated as valid floats.
theta_info.validate_value(np.array(np.inf))

In [8]:
# Valid values for Sigma.
Sigma_values = [np.diag([1.0,1.0]), np.array([1,.9,.9,1]).reshape((2,2))]

for val in Sigma_values:
    Sigma_info.validate_value(val)

In [9]:
# Invalid: not numpy array.
theta_info.validate_value(12.0)

TypeError: Value must be a numpy.ndarray, got <class 'float'>

In [10]:
# Invalid dtype.
theta_info.validate_value(np.array([1]))

TypeError: Value must have numpy.floating dtype, not int64

In [11]:
# Invalid shape.
Sigma_info.validate_value(np.arange(4, dtype=float))

TypeError: Param has shape (2, 2), but value has shape (4,)

In [12]:
# Bound constraint violation.
theta_info.validate_value(np.array(-1.0))

ValueError: Value has 1 lower bound violations.

In [13]:
# PSD constraint violation.
Sigma_info.validate_value(np.diag([1.0, -0.1]))

ValueError: Value violates positive semidefinite constraint.

In [14]:
# Simplex constraint violation.
example_info = ParamInfo(value_type="float", shape=(3,), constraint="simplex")
example_info.validate_value(np.array([0.8, 0.2, 0.1]))

ValueError: Violation of simplex sum-to-one constraint.

In [15]:
example_info.validate_value(np.array([0.8, 0.2, 0.0]))

ValueError: Violation of simplex sum-to-one constraint.

## Other basic properties

In [ ]:
# Use NoConstraint for unconstrained parameters.
example_info = ParamInfo(value_type="int", shape=(5,4,2), constraint=NoConstraint)
print(example_info)

In [ ]:
# Infinite values can alternatively be provided to produce one-sided bounds, but they will be converted to None.
example_info = ParamInfo(value_type="int", shape=(5,4,2), constraint=(-1, np.inf))
print(example_info)

In [ ]:
# If bounds tuple is provided but does not impose any constraint, will be simplified to NoConstraint.
example_info = ParamInfo(value_type="int", shape=(5,4,2), constraint=(-np.inf, np.inf))
print(example_info)

In [ ]:
# Simplex constraint is for parameters whose values are in [0,1] and sum to 1.
example_info = ParamInfo(value_type="float", shape=(3,), constraint="simplex")
print(example_info)

## Argument Checking

### Invalid Types

In [ ]:
# Invalid type
example_info = ParamInfo(value_type="not_a_type", shape=(5,), constraint=NoConstraint)

### Invalid shape

In [ ]:
# Shape is not consistent with constraint.
example_info = ParamInfo(value_type="float", shape=(5,2), constraint="psd")

### Invalid constraints

In [ ]:
# Unrecognized constraint.
example_info = ParamInfo(value_type="int", shape=(1,), constraint="not_a_constraint")

In [ ]:
# Bound constraint with lower bound larger than upper bound.
example_info = ParamInfo(value_type="float", shape=(9,1), constraint=(1,0))

In [ ]:
# Bound constraint of invalid dimension.
example_info = ParamInfo(value_type="float", shape=(9,1), constraint=(0,1,4))

In [ ]:
# Simplex constraint only allowed when value_type is float.
example_info = ParamInfo(value_type="int", shape=(3,), constraint="simplex")

In [ ]:
# For absence of constraint, use NoConstraint, not None. The latter is reserved for missing argument values.
example_info = ParamInfo(value_type="int", shape=(3,), constraint=None)

# ParamGroup Class
Let's suppose we are interested in the parameters $\theta \in [0,\infty)$ and $\Sigma \in \mathbb{R}^{2 \times 2}$. The former is a scalar constrained to be nonnegative and the latter is a positive semidefinite (PSD) matrix. We will create a `ParamGroup` object that encodes the parameter set $\{\theta, \Sigma\}$.

### Instantiating ParamGroup

The ParamGroup class only accepts a dictionary where the keys are strings and values are `ParamInfo` objects. 

In [ ]:
# Create ParamInfo objects.
theta_info = ParamInfo(value_type="float", shape=(), constraint=(0, None))
Sigma_info = ParamInfo(value_type="float", shape=(2,2), constraint="psd")

In [ ]:
# Incorrect: only string keys allowed.
param_info = {1: theta_info, 2: Sigma_info}
param_group = ParamGroup(param_info)

In [ ]:
# Incorrect: dict values must be ParamInfo objects.
param_info = {
    "theta": {"value_type": "float", "shape": (), "constraint": (0, None)}, 
    "Sigma": {"value_type": "float", "shape": (2,2), "constraint": "psd"}}
param_group = ParamGroup(param_info)

In [ ]:
# Correct
param_info = {"theta": theta_info, "Sigma": Sigma_info}
param_group = ParamGroup(param_info)

### ParamGroup Functionality

In [ ]:
# Print parameter group
print(param_group)

In [ ]:
# To prevent manipulation from affecting comparison, need to do a deep copy
param_group_comparison = ParamGroup(copy.deepcopy(param_info))
param_group == param_group_comparison

In [ ]:
# Get length of parameter group
len(param_group)

In [ ]:
# Parameter names (orders alphabetically)
param_group.get_param_names()

In [ ]:
# Flattened parameter names (i.e., every scalar value gets a name). Flattened names list has length
# equal to `len(param_group)`. Orders first by alphabetical unflattened parameter names, and second 
# by row major order within multivariate parameters. Names for the individual scalar elements of 
# multivariate parameters are autogenerated based on the element index in the array.
param_group.get_param_names(flatten=True)

In [ ]:
# Add new parameter.
new_param_info = ParamInfo(value_type="int", shape=(5,), constraint=(NoConstraint))

param_group.add_param("new_param_name", new_param_info)
print(param_group)

In [ ]:
# Remove new parameter and theta 
param_group.remove_param(["new_param_name", "theta"])
print(param_group)

In [ ]:
# Compare after changes
param_group == param_group_comparison

In [ ]:
# Another example of comparison

param1 = ParamInfo(value_type="float", shape=(2, 2), constraint="psd")
param2 = ParamInfo(value_type="int", shape=(3,), constraint=(0, 10))

group1 = ParamGroup({"alpha": param1, "beta": param2})
group2 = ParamGroup({"alpha": param1, "beta": param2})

print(group1 == group2)  # True, since all parameters match

# Modify one parameter
group3 = ParamGroup({"alpha": param1, "beta": ParamInfo(value_type="float", shape=(3,), constraint=(0, 10))})

print(group1 == group3)  # False, since beta's type changed from int to float


# ParamValue Class
The `ParamValue` class encodes the notion of a specific value a parameter can assume. For the example above, an example of a valid value might be 
$$
\left\{1.0, \begin{bmatrix} 1.0 & 0.0 \\ 0.0 & 1.0 \end{bmatrix} \right\}.
$$

The core of the class is a dictionary with keys corresponding to parameter names and values to the associated parameter values. The class ensures that a given value is of the correct type and size, and satisfies all of the constraints as specified in a `ParamGroup` object. It also features a `to_array` method that converts the parameter value dictionary to an array, with the elements sorted in a canonical order as determined by `ParamGroup`.


In [7]:
# Create ParamInfo objects for the parameters.
theta_info = ParamInfo(value_type="float", shape=(), constraint=(0, None))
Sigma_info = ParamInfo(value_type="float", shape=(2, 2), constraint="psd")

# Create a dictionary mapping parameter names to their metadata.
param_info = {"theta": theta_info, "Sigma": Sigma_info}
param_group = ParamGroup(param_info)

# Define initial values for the parameters.
theta_val = np.array(1.0)
Sigma_val = np.array([[1.0, 0.0],
                      [0.0, 1.0]])

# Create the values dictionary.
values = {"theta": theta_val, "Sigma": Sigma_val}

# Create an instance of ParamGroupValues with the parameter group and values.
param_group_values = ParamGroupValues(param_group, values)

# Use the to_array function to flatten all parameter values.
flat_array = param_group_values.to_array()

print(flat_array)

[1. 0. 0. 1. 1.]
